In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as T
import albumentations as A

# Enable cuDNN auto‑tuner for the current configuration (won’t affect correctness)
torch.backends.cudnn.benchmark = True




In [2]:
BASE_INPUT = "/kaggle/input/cassava-leaf-disease-classification"

sample_sub_path = os.path.join(BASE_INPUT, "sample_submission.csv")
test_images_path = os.path.join(BASE_INPUT, "test_images")
train_images_path = os.path.join(BASE_INPUT, "train_images")
train_csv_path = os.path.join(BASE_INPUT, "train.csv")




In [3]:
model = models.resnext50_32x4d(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 5)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)




/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNeXt50_32X4D_Weights.IMAGENET1K_V1`. You can also use `weights=ResNeXt50_32X4D_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnext50_32x4d-7cdf4587.pth" to /root/.cache/torch/hub/checkpoints/resnext50_32x4d-7cdf4587.pth
100%|██████████| 95.8M/95.8M [00:12<00:00, 7.92MB/s]


In [4]:
class CassavaDataset(Dataset):
    def __init__(self, csv_path, img_dir, transforms):
        self.df = pd.read_csv(csv_path)
        self.img_dir = img_dir
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row["image_id"])
        image = np.array(Image.open(img_path).convert("RGB"))
        aug = self.transforms(image=image)
        aug_image = aug["image"]  # already normalized
        tensor_image = torch.from_numpy(aug_image).permute(2, 0, 1).float()
        label = int(row["label"])
        return tensor_image, label


train_aug = A.Compose(
    [
        A.RandomResizedCrop(size=(256, 256), scale=(0.8, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
            max_pixel_value=255.0,
        ),
    ]
)

train_dataset = CassavaDataset(train_csv_path, train_images_path, train_aug)
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=4,  # increased from 2 to better utilize CPU cores
    pin_memory=True,
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

model.train()
for epoch in range(5):
    epoch_loss = 0.0
    for imgs, labels in train_loader:
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * imgs.size(0)

    avg_loss = epoch_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/5 - Loss: {avg_loss:.4f}")

model.eval()




Epoch 1/5 - Loss: 0.5314
Epoch 2/5 - Loss: 0.3614
Epoch 3/5 - Loss: 0.2784
Epoch 4/5 - Loss: 0.2103
Epoch 5/5 - Loss: 0.1565


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
      (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(128, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1

In [5]:
sub_aug = A.Compose(
    [
        A.Resize(256, 256),
        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
            max_pixel_value=255.0,
        ),
    ]
)

sample_sub = pd.read_csv(sample_sub_path)

predictions = []
# Iterate over image ids without using iterrows (faster)
for image_id in sample_sub["image_id"].values:
    img_path = os.path.join(test_images_path, image_id)
    # Load image once
    base_image = np.array(Image.open(img_path).convert("RGB"))

    # Generate 10 augmented versions in a list
    aug_images = []
    for _ in range(10):
        aug = sub_aug(image=base_image)["image"]
        tensor_img = torch.from_numpy(aug).permute(2, 0, 1).float()
        aug_images.append(tensor_img)

    # Stack into a batch and move to device once
    batch_tensor = torch.stack(aug_images).to(
        device, non_blocking=True
    )  # shape (10, 3, 256, 256)

    with torch.no_grad():
        outputs = model(batch_tensor)  # shape (10, 5)
    image_pred = outputs.mean(dim=0)  # average TTA predictions

    _, pred_label = torch.max(image_pred, dim=0)
    predictions.append([image_id, int(pred_label.item())])

sub_df = pd.DataFrame(predictions, columns=["image_id", "label"])
sub_df.to_csv("submission.csv", index=False)
print(sub_df.head())

         image_id  label
0  1234294272.jpg      3
1  1234332763.jpg      3
2  1234375577.jpg      3
3  1234555380.jpg      1
4  1234571117.jpg      3
